# Mission 10 · Stage 6
## Colab 공식 test 1회 전용 노트북

Kaggle OOF에서 최종 설정을 고정했습니다.

| 모델 | Accuracy | Macro F1 | `talk.religion.misc` F1 |
|---|---:|---:|---:|
| **Subject + Body LinearSVC** | **0.9031** | **0.8999** | **0.7818** |
| Body-only LinearSVC | 0.7643 | 0.7559 | 0.4355 |
| Frozen ModernBERT + LinearSVC | 0.7254 | 0.6992 | 0.1096 |

이 노트북은 모델 탐색이나 OOF를 다시 돌리지 않습니다.

```text
Drive 완료 여부 확인
→ CV pool 16,019개 전체로 고정 모델 최종 학습
→ official test 2,827개 한 번 평가
→ 결과와 완료 lock을 Drive에 저장
```

공유 Drive의 `official_test_done.lock` 또는 `final_test_result.json`이 이미 있으면 test를 다시 계산하지 않습니다.


## 런타임

최종 모델은 Transformer가 아니라 LinearSVC이므로 GPU가 필요 없습니다.

```text
Colab 런타임: CPU
하드웨어 가속기: 없음
```


In [1]:
!pip install -q -U scikit-learn google-api-python-client


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.4 MB/s eta 0:00:00


In [2]:
import hashlib
import io
import json
import os
import re
import sys
import warnings
from datetime import datetime, timezone
from email.header import decode_header, make_header
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

SEED = 42
TEST_SIZE = 0.15
WORD_MAX_FEATURES = 80_000
CHAR_MAX_FEATURES = 120_000

STAGE6_FOLDER_ID = '1zPDPLjC_6-5qVacMAwTtFCfQc39lL6x1'
RESULTS_FOLDER_ID = '1Q9LqghSLHtpFtQiUVz4ZAjvmZ9QzRE1q'

RUNNING_LOCK_NAME = 'official_test_running.lock'
DONE_LOCK_NAME = 'official_test_done.lock'
FINAL_RESULT_NAME = 'final_test_result.json'
FINAL_REPORT_NAME = 'final_test_class_report.csv'
FINAL_CONFUSIONS_NAME = 'final_test_top_confusions.csv'
FINAL_PROBABILITIES_NAME = 'final_test_probabilities.npy'

FORCE_REMOVE_RUNNING_LOCK = False
SAVE_MODEL_TO_DRIVE = False

LOCKED_CONFIG = {
    'input':'subject_body',
    'model':'linear_svc',
    'word_ngram_range':[1,2],
    'char_ngram_range':[3,5],
    'word_max_features':WORD_MAX_FEATURES,
    'char_max_features':CHAR_MAX_FEATURES,
    'C':1.0,
    'seed':SEED,
    'test_size':TEST_SIZE,
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(LOCKED_CONFIG, sort_keys=True).encode('utf-8')
).hexdigest()
print('Python:', sys.version.split()[0])
print('Locked config hash:', CONFIG_HASH)


Python: 3.12.13
Locked config hash: 463bb1d53b56f6ae0e4cb00bccc1280405612baf30247248bcff9a2bd925439e


## Google Drive 인증

폴더 경로가 아니라 기존 Stage 6 Drive 폴더 ID를 사용합니다. 따라서 My Drive 안에서 폴더를 옮겨도 동일한 결과 폴더를 찾습니다.


In [3]:
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload

auth.authenticate_user()
credentials, _ = default()
drive_service = build('drive', 'v3', credentials=credentials)

def esc(value):
    return str(value).replace('\\','\\\\').replace("'", "\\'")

def find_file(name, parent_id=RESULTS_FOLDER_ID):
    q = f"'{esc(parent_id)}' in parents and name = '{esc(name)}' and trashed = false"
    res = drive_service.files().list(
        q=q, spaces='drive',
        fields='files(id,name,mimeType,createdTime,modifiedTime,size)',
        orderBy='modifiedTime desc', pageSize=10,
    ).execute()
    files = res.get('files', [])
    return files[0] if files else None

def download_bytes(file_id):
    req = drive_service.files().get_media(fileId=file_id)
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, req)
    done = False
    while not done:
        _, done = dl.next_chunk()
    return buf.getvalue()

def download_json(file_id):
    return json.loads(download_bytes(file_id).decode('utf-8'))

def upload_bytes(name, content, mime_type, replace=True, parent_id=RESULTS_FOLDER_ID):
    existing = find_file(name, parent_id)
    media = MediaIoBaseUpload(io.BytesIO(content), mimetype=mime_type, resumable=False)
    if existing and replace:
        return drive_service.files().update(
            fileId=existing['id'], media_body=media,
            fields='id,name,modifiedTime'
        ).execute()
    return drive_service.files().create(
        body={'name':name, 'parents':[parent_id]},
        media_body=media,
        fields='id,name,modifiedTime'
    ).execute()

def upload_text(name, text, mime_type='text/plain', replace=True):
    return upload_bytes(name, text.encode('utf-8'), mime_type, replace)

def delete_file(file_id):
    drive_service.files().delete(fileId=file_id).execute()

print('Drive 인증 완료')


Drive 인증 완료


In [4]:
def read_existing_result():
    result_file = find_file(FINAL_RESULT_NAME)
    done_lock = find_file(DONE_LOCK_NAME)
    if result_file:
        result = download_json(result_file['id'])
        saved_hash = result.get('config_hash')
        if saved_hash and saved_hash != CONFIG_HASH:
            raise RuntimeError('기존 test 결과의 config hash가 현재 설정과 다릅니다. 덮어쓰지 않습니다.')
        print('이미 완료된 official test 결과를 불러왔습니다.')
        display(pd.DataFrame([{
            **result['test_metrics'],
            'misc_precision':result['talk.religion.misc']['precision'],
            'misc_recall':result['talk.religion.misc']['recall'],
            'misc_f1':result['talk.religion.misc']['f1'],
        }]).style.format(precision=4))
        return result
    if done_lock:
        raise RuntimeError('완료 lock은 있지만 final_test_result.json이 없습니다.')
    return None

def acquire_lock():
    running = find_file(RUNNING_LOCK_NAME)
    if running:
        if FORCE_REMOVE_RUNNING_LOCK:
            delete_file(running['id'])
            print('기존 running lock 제거:', running['modifiedTime'])
        else:
            raise RuntimeError(
                'official_test_running.lock이 있습니다. 다른 Kaggle/Colab 실행이 진행 중이거나 '
                '이전 세션이 중단됐습니다. 실행 중인 세션이 없음을 확인한 뒤 '
                'FORCE_REMOVE_RUNNING_LOCK=True로 한 번만 재실행하세요.'
            )
    payload = {
        'status':'running', 'config_hash':CONFIG_HASH,
        'started_at_utc':datetime.now(timezone.utc).isoformat(),
    }
    created = upload_text(
        RUNNING_LOCK_NAME, json.dumps(payload, ensure_ascii=False, indent=2),
        mime_type='application/json', replace=False,
    )
    print('Official test lock 생성:', created['id'])
    return created['id']

def release_lock():
    running = find_file(RUNNING_LOCK_NAME)
    if running:
        delete_file(running['id'])
        print('Running lock 제거 완료')


In [5]:
HEADER_BODY_SEPARATOR = re.compile(r'\r?\n\r?\n', flags=re.MULTILINE)
REPLY_PREFIX = re.compile(r'^\s*((re|fw|fwd)\s*:\s*)+', flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'\b[\w.\-+]+@[\w.\-]+\.\w+\b')
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)

def decode_mime(value):
    try:
        return str(make_header(decode_header(value)))
    except Exception:
        return str(value)

def extract_subject(raw_text, target_names):
    header = HEADER_BODY_SEPARATOR.split(str(raw_text), maxsplit=1)[0]
    header = re.sub(r'\r?\n[ \t]+', ' ', header)
    subject = ''
    for line in header.splitlines():
        if line.lower().startswith('subject:'):
            subject = line.split(':', 1)[1].strip()
            break
    subject = decode_mime(subject)
    subject = REPLY_PREFIX.sub('', subject)
    subject = EMAIL_PATTERN.sub(' ', subject)
    subject = URL_PATTERN.sub(' ', subject)
    for class_name in target_names:
        subject = re.sub(re.escape(class_name), ' ', subject, flags=re.IGNORECASE)
    subject = re.sub(r'\s+', ' ', subject).strip()
    return subject if subject else '[NO_SUBJECT]'

def load_fixed_split():
    raw = fetch_20newsgroups(subset='all', remove=(), shuffle=True, random_state=SEED)
    body = fetch_20newsgroups(
        subset='all', remove=('headers','footers','quotes'),
        shuffle=True, random_state=SEED,
    )
    labels = np.asarray(raw.target, dtype=np.int64)
    body_labels = np.asarray(body.target, dtype=np.int64)
    if not np.array_equal(labels, body_labels):
        raise RuntimeError('Raw/body 데이터 순서 불일치')
    names = list(raw.target_names)
    subjects = [extract_subject(text, names) for text in raw.data]
    texts = np.asarray([
        f'[SUBJECT]\n{s}\n\n[BODY]\n{b}' for s, b in zip(subjects, body.data)
    ], dtype=object)
    idx = np.arange(len(labels))
    cv_idx, test_idx = train_test_split(
        idx, test_size=TEST_SIZE, stratify=labels, random_state=SEED
    )
    split_hash = hashlib.sha256(np.asarray(test_idx, dtype=np.int64).tobytes()).hexdigest()
    return {
        'cv_texts':texts[cv_idx], 'cv_labels':labels[cv_idx],
        'test_texts':texts[test_idx], 'test_labels':labels[test_idx],
        'target_names':names, 'test_indices_hash':split_hash,
    }


In [6]:
def build_vectorizer():
    return FeatureUnion([
        ('word', TfidfVectorizer(
            analyzer='word', ngram_range=(1,2), min_df=2, max_df=0.98,
            sublinear_tf=True, max_features=WORD_MAX_FEATURES, dtype=np.float32,
        )),
        ('char', TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3,5), min_df=2,
            sublinear_tf=True, max_features=CHAR_MAX_FEATURES, dtype=np.float32,
        )),
    ], n_jobs=min(2, os.cpu_count() or 1))

def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    return {
        'accuracy':float(accuracy_score(y_true, y_pred)),
        'macro_precision':float(p), 'macro_recall':float(r), 'macro_f1':float(f),
    }

def class_metrics(y_true, y_pred, class_id, class_name):
    yt = (y_true == class_id).astype(np.int64)
    yp = (y_pred == class_id).astype(np.int64)
    p, r, f, _ = precision_recall_fscore_support(
        yt, yp, average='binary', zero_division=0
    )
    return {
        'class':class_name, 'precision':float(p), 'recall':float(r),
        'f1':float(f), 'support':int(yt.sum()),
    }

def top_confusions(y_true, y_pred, names):
    cm = confusion_matrix(y_true, y_pred)
    rows=[]
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j and cm[i,j]:
                rows.append({'true_class':names[i], 'predicted_class':names[j], 'count':int(cm[i,j])})
    return pd.DataFrame(rows).sort_values('count', ascending=False).reset_index(drop=True)

def score_to_prob(scores):
    scores = np.asarray(scores, dtype=np.float64)
    scores -= scores.max(axis=1, keepdims=True)
    exp = np.exp(scores)
    return (exp / exp.sum(axis=1, keepdims=True)).astype(np.float32)


# Official test 1회 실행

이 셀을 다시 실행해도 Drive의 완료 파일을 먼저 확인하므로 실제 test 계산은 반복되지 않습니다.


In [7]:
def run_official_test_once():
    existing = read_existing_result()
    if existing is not None:
        return existing
    acquire_lock()
    try:
        data = load_fixed_split()
        print('CV pool:', len(data['cv_labels']))
        print('Official test:', len(data['test_labels']))
        print('Test split hash:', data['test_indices_hash'])

        vectorizer = build_vectorizer()
        print('CV pool 전체 TF-IDF 학습...')
        x_train = vectorizer.fit_transform(data['cv_texts'])
        print('Official test TF-IDF 변환...')
        x_test = vectorizer.transform(data['test_texts'])

        classifier = LinearSVC(
            C=1.0, dual=True, max_iter=5000, tol=1e-4,
            random_state=SEED + 100_000,
        )
        print('CV pool 전체 LinearSVC 학습...')
        classifier.fit(x_train, data['cv_labels'])
        print('Official test 추론 및 1회 평가...')
        scores = classifier.decision_function(x_test)
        pred = scores.argmax(axis=1)
        prob = score_to_prob(scores)

        test_metrics = metrics(data['test_labels'], pred)
        misc_name = 'talk.religion.misc'
        misc_id = data['target_names'].index(misc_name)
        misc = class_metrics(data['test_labels'], pred, misc_id, misc_name)
        report = pd.DataFrame(classification_report(
            data['test_labels'], pred, target_names=data['target_names'],
            output_dict=True, zero_division=0,
        )).T
        confusions = top_confusions(data['test_labels'], pred, data['target_names'])

        result = {
            'status':'complete',
            'completed_at_utc':datetime.now(timezone.utc).isoformat(),
            'config_hash':CONFIG_HASH,
            'locked_config':LOCKED_CONFIG,
            'test_indices_hash':data['test_indices_hash'],
            'documents':int(len(data['test_labels'])),
            'test_metrics':test_metrics,
            'talk.religion.misc':misc,
        }

        upload_text(FINAL_RESULT_NAME, json.dumps(result, ensure_ascii=False, indent=2), 'application/json')
        upload_text(FINAL_REPORT_NAME, report.to_csv(), 'text/csv')
        upload_text(FINAL_CONFUSIONS_NAME, confusions.to_csv(index=False), 'text/csv')
        buf = io.BytesIO(); np.save(buf, prob)
        upload_bytes(FINAL_PROBABILITIES_NAME, buf.getvalue(), 'application/octet-stream')

        if SAVE_MODEL_TO_DRIVE:
            local = Path('/content/stage6_final_model.joblib')
            joblib.dump({
                'vectorizer':vectorizer, 'classifier':classifier,
                'target_names':data['target_names'], 'config':LOCKED_CONFIG,
            }, local, compress=3)
            upload_bytes('stage6_final_model.joblib', local.read_bytes(), 'application/octet-stream')

        done_payload = {
            'status':'done', 'config_hash':CONFIG_HASH,
            'test_indices_hash':data['test_indices_hash'],
            'final_result_file':FINAL_RESULT_NAME,
            'completed_at_utc':result['completed_at_utc'],
        }
        upload_text(DONE_LOCK_NAME, json.dumps(done_payload, ensure_ascii=False, indent=2), 'application/json')
        release_lock()

        print(json.dumps(result, ensure_ascii=False, indent=2))
        display(pd.DataFrame([{
            **test_metrics,
            'misc_precision':misc['precision'],
            'misc_recall':misc['recall'],
            'misc_f1':misc['f1'],
        }]).style.format(precision=4))
        return result
    except Exception:
        release_lock()
        raise

FINAL_TEST_RESULT = run_official_test_once()


Official test lock 생성: 1D4rKAVA4Ra_LKJ8sFgsFt7PwBbUcjb5c
CV pool: 16019
Official test: 2827
Test split hash: 1703cf32c5a967a3e5d58b3639c513429c1800871d10d887cf639a0ca86fe5f5
CV pool 전체 TF-IDF 학습...
Official test TF-IDF 변환...
CV pool 전체 LinearSVC 학습...
Official test 추론 및 1회 평가...
Running lock 제거 완료
{
  "status": "complete",
  "completed_at_utc": "2026-07-27T04:41:17.016848+00:00",
  "config_hash": "463bb1d53b56f6ae0e4cb00bccc1280405612baf30247248bcff9a2bd925439e",
  "locked_config": {
    "input": "subject_body",
    "model": "linear_svc",
    "word_ngram_range": [
      1,
      2
    ],
    "char_ngram_range": [
      3,
      5
    ],
    "word_max_features": 80000,
    "char_max_features": 120000,
    "C": 1.0,
    "seed": 42,
    "test_size": 0.15
  },
  "test_indices_hash": "1703cf32c5a967a3e5d58b3639c513429c1800871d10d887cf639a0ca86fe5f5",
  "documents": 2827,
  "test_metrics": {
    "accuracy": 0.9066147859922179,
    "macro_precision": 0.9082191315480582,
    "macro_recall": 0.

,accuracy,macro_precision,macro_recall,macro_f1,misc_precision,misc_recall,misc_f1
0,0.9066,0.9082,0.9029,0.9046,0.9359,0.7766,0.8488


## 재실행 규칙

- 정상 완료 후 다시 실행하면 기존 Drive 결과만 표시합니다.
- 기존 Kaggle Stage 6 노트북에서는 official test를 다시 실행하지 않습니다.
- Colab이 강제 종료되어 `official_test_running.lock`만 남고 다른 실행이 없을 때만 `FORCE_REMOVE_RUNNING_LOCK=True`로 바꿔 재실행합니다.
